# nanoGPT Scaling Study (Colab)

**Before you start:** Runtime > Change runtime type > **GPU (T4)**. Then run the cells in order.

What you will do: (1) write the project files, (2) run sanity tests, (3) run one quick training, (4) run the scaling sweep, (5) plot and save results.

If Colab disconnects, re-run the first cells and then the sweep cell: finished runs are skipped.

In [ ]:
import torch
import platform
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__, "- CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))


## Optional: save results to Google Drive
Run this if you want results to survive a disconnect. Set `RESULTS_DIR` to a Drive folder.

In [ ]:
import os
USE_DRIVE = False   # set True to save results to Google Drive
RESULTS_DIR = 'results'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = '/content/drive/MyDrive/nanogpt_results'
os.makedirs('tests', exist_ok=True)
print('Results will be saved to', RESULTS_DIR)

## 1. Project files
Each cell below writes one file. Read them: you need to be able to explain every line in an interview.

In [ ]:
%%writefile data.py
"""Character-level dataset and preprocessing for nanoGPT (Tiny Shakespeare by default).

Responsibilities:
- Reliable, atomic dataset download with SHA-256 verification.
- Deterministic character-level tokenization (stoi, itos).
- Sequential train / validation split.
- Deterministic batch generation using explicit torch.Generator.
- Strict error handling for out-of-vocabulary characters.
"""
import hashlib
import os
import urllib.request

import torch

TINY_SHAKESPEARE_URL = (
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
)
TINY_SHAKESPEARE_SHA256 = "4b6807d90d796e9526e033d5ba4b54e38c92a543f076b1580cae43b185e597c5"


def compute_sha256(text: str) -> str:
    """Returns SHA-256 hexadecimal digest of UTF-8 encoded text."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def get_dataset_metadata(text: str) -> dict:
    """Computes basic structural metadata for a text corpus."""
    chars = sorted(set(text))
    return {
        "character_count": len(text),
        "byte_count": len(text.encode("utf-8")),
        "vocab_size": len(chars),
        "sha256": compute_sha256(text),
    }


def load_text(
    path: str = "data/input.txt",
    url: str = TINY_SHAKESPEARE_URL,
    verify_sha256: bool = False,
    expected_sha256: str | None = None,
) -> str:
    """Load the text corpus, downloading atomically to a temporary file on first use."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    if not os.path.exists(path):
        tmp_path = path + ".tmp"
        try:
            urllib.request.urlretrieve(url, tmp_path)
            with open(tmp_path, encoding="utf-8") as f:
                content = f.read()
            if verify_sha256 and expected_sha256:
                actual_hash = compute_sha256(content)
                if actual_hash != expected_sha256:
                    raise ValueError(
                        f"Dataset SHA-256 mismatch!\nExpected: {expected_sha256}\nActual:   {actual_hash}"
                    )
            os.replace(tmp_path, path)
            return content
        except Exception as e:
            if os.path.exists(tmp_path):
                try:
                    os.remove(tmp_path)
                except OSError:
                    pass
            raise RuntimeError(
                f"Could not load or download dataset ({e}).\n"
                f"Please manually download from:\n{url}\nand save as: {path}"
            ) from e

    with open(path, encoding="utf-8") as f:
        content = f.read()

    if verify_sha256 and expected_sha256:
        actual_hash = compute_sha256(content)
        if actual_hash != expected_sha256:
            raise ValueError(
                f"Dataset file at {path} has corrupted SHA-256!\nExpected: {expected_sha256}\nActual:   {actual_hash}"
            )
    return content


class CharDataset:
    """Character-level tokenizer and training dataset."""

    def __init__(self, text: str, val_frac: float = 0.1):
        if not isinstance(text, str) or len(text) == 0:
            raise ValueError("text must be a non-empty string")
        if not (0.0 < val_frac < 1.0):
            raise ValueError(f"val_frac must be in the open interval (0.0, 1.0), got {val_frac}")

        chars = sorted(set(text))
        self.chars = chars
        self.stoi = {c: i for i, c in enumerate(chars)}
        self.itos = {i: c for i, c in enumerate(chars)}
        self.vocab_size = len(chars)

        data = torch.tensor([self.stoi[c] for c in text], dtype=torch.long)
        n = int(len(data) * (1 - val_frac))
        self.train, self.val = data[:n], data[n:]

        if len(self.train) == 0 or len(self.val) == 0:
            raise ValueError(
                f"Dataset split resulted in empty slice: train length {len(self.train)}, val length {len(self.val)}"
            )

    def encode(self, s: str) -> list[int]:
        """Encodes string to integer token IDs. Raises ValueError on unknown characters."""
        unknown = [c for c in s if c not in self.stoi]
        if unknown:
            unique_unk = sorted(set(unknown))
            raise ValueError(
                f"Unsupported character(s) in prompt: {unique_unk!r}. "
                f"Vocabulary only contains {self.vocab_size} characters: {self.chars!r}"
            )
        return [self.stoi[c] for c in s]

    def decode(self, ids: list[int] | torch.Tensor) -> str:
        """Decodes integer token IDs back to a string."""
        return "".join(self.itos[int(i)] for i in ids)

    def get_batch(
        self,
        split: str,
        batch_size: int,
        block_size: int,
        device: str | torch.device,
        generator: torch.Generator | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Samples a batch of inputs x and targets y with y shifted by +1."""
        if split not in ("train", "val"):
            raise ValueError(f"split must be 'train' or 'val', got {split!r}")
        if batch_size <= 0:
            raise ValueError(f"batch_size must be positive, got {batch_size}")
        if block_size <= 0:
            raise ValueError(f"block_size must be positive, got {block_size}")

        d = self.train if split == "train" else self.val
        if len(d) <= block_size:
            raise ValueError(
                f"Dataset slice '{split}' has length {len(d)}, which is <= block_size {block_size}"
            )

        high = len(d) - block_size
        ix = torch.randint(high, (batch_size,), generator=generator)
        x = torch.stack([d[i : i + block_size] for i in ix])
        y = torch.stack([d[i + 1 : i + 1 + block_size] for i in ix])
        return x.to(device), y.to(device)


In [ ]:
%%writefile model.py
"""A small GPT (decoder-only transformer) written from scratch in PyTorch.

Architecture details:
- Learned token and absolute positional embeddings.
- Multi-head causal self-attention with fused QKV projection.
- F.scaled_dot_product_attention with is_causal=True.
- MLP with 4x expansion and GELU activation.
- Pre-LN and Post-LN architectural variants.
- Weight tying between token embeddings (wte) and language model head.
- Scaled residual projection initialization (0.02 / sqrt(2 * n_layer)).
"""
import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class GPTConfig:
    block_size: int = 128
    vocab_size: int = 65
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    dropout: float = 0.1
    ln_placement: str = "pre"  # "pre" (LayerNorm before sublayers) or "post" (original Transformer)

    def __post_init__(self):
        if self.block_size <= 0:
            raise ValueError(f"block_size must be positive, got {self.block_size}")
        if self.vocab_size <= 0:
            raise ValueError(f"vocab_size must be positive, got {self.vocab_size}")
        if self.n_layer <= 0:
            raise ValueError(f"n_layer must be positive, got {self.n_layer}")
        if self.n_head <= 0:
            raise ValueError(f"n_head must be positive, got {self.n_head}")
        if self.n_embd <= 0:
            raise ValueError(f"n_embd must be positive, got {self.n_embd}")
        if self.n_embd % self.n_head != 0:
            raise ValueError(f"n_embd ({self.n_embd}) must be divisible by n_head ({self.n_head})")
        if not (0.0 <= self.dropout < 1.0):
            raise ValueError(f"dropout must be in [0.0, 1.0), got {self.dropout}")
        if self.ln_placement not in ("pre", "post"):
            raise ValueError(f"ln_placement must be 'pre' or 'post', got {self.ln_placement!r}")

    def to_dict(self):
        return asdict(self)


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        if cfg.n_embd % cfg.n_head != 0:
            raise ValueError(f"n_embd ({cfg.n_embd}) must be divisible by n_head ({cfg.n_head})")
        self.n_head = cfg.n_head
        self.n_embd = cfg.n_embd
        self.head_dim = cfg.n_embd // cfg.n_head
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd)
        self.attn_dropout = cfg.dropout
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, T, C) -> (B, n_head, T, head_dim)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        # Each position may only attend to itself and earlier positions (is_causal=True).
        y = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.attn_dropout if self.training else 0.0, is_causal=True
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.proj(y))


class MLP(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd)
        self.proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.drop(self.proj(F.gelu(self.fc(x))))


class Block(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        if cfg.ln_placement not in ("pre", "post"):
            raise ValueError(f"ln_placement must be 'pre' or 'post', got {cfg.ln_placement!r}")
        self.pre = cfg.ln_placement == "pre"
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.pre:
            x = x + self.attn(self.ln1(x))
            x = x + self.mlp(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.mlp(x))
        return x


class GPT(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd) if cfg.ln_placement == "pre" else nn.Identity()
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.wte.weight  # weight tying
        self.apply(self._init_weights)
        # scaled init for residual projections (GPT-2 style)
        for name, p in self.named_parameters():
            if name.endswith("proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    @staticmethod
    def _init_weights(m: nn.Module):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def num_params(self, non_embedding: bool = False) -> int:
        """Returns total unique parameters. Tied weights are counted once."""
        n = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n -= self.wte.weight.numel() + self.wpe.weight.numel()
        return n

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        B, T = idx.shape
        if T > self.cfg.block_size:
            raise ValueError(f"Sequence length {T} exceeds model block_size {self.cfg.block_size}")
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.wte(idx) + self.wpe(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            if targets.shape != idx.shape:
                raise ValueError(f"targets shape {targets.shape} does not match idx shape {idx.shape}")
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: int | None = None,
    ) -> torch.Tensor:
        """Autoregressively sample tokens. Call model.eval() before generation."""
        if temperature <= 0.0:
            raise ValueError(f"temperature must be positive, got {temperature}")
        if max_new_tokens < 0:
            raise ValueError(f"max_new_tokens must be non-negative, got {max_new_tokens}")
        if top_k is not None and top_k <= 0:
            raise ValueError(f"top_k must be positive if specified, got {top_k}")

        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.cfg.block_size else idx[:, -self.cfg.block_size :]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


In [ ]:
%%writefile train.py
"""Training loop for nanoGPT with AdamW, cosine LR schedule, mixed precision, and step-level resumption.

Supports execution as:
- A Python library: train(**overrides)
- A command-line script: python train.py --n_layer 4 --lr 1e-3
"""
import argparse
import hashlib
import json
import math
import os
import platform
import random
import time
from contextlib import nullcontext

import torch

from data import CharDataset, compute_sha256, load_text
from model import GPT, GPTConfig

DEFAULTS = dict(
    block_size=128,
    batch_size=64,
    n_layer=4,
    n_head=4,
    n_embd=128,
    dropout=0.1,
    ln_placement="pre",
    max_iters=3000,
    eval_interval=250,
    eval_iters=50,
    lr=1e-3,
    min_lr=1e-4,
    warmup=100,
    weight_decay=0.1,
    grad_clip=1.0,
    seed=0,
    out_dir="out/run",
    save_ckpt=True,
    save_resume=True,
    resume_from=None,
    dataset_path="data/input.txt",
    data_text=None,
    device=None,
)


def seed_everything(seed: int):
    """Seed Python, random, NumPy (if present), PyTorch CPU and all CUDA devices."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    try:
        import numpy as np

        np.random.seed(seed)
    except ImportError:
        pass


def get_lr(it: int, max_iters: int, lr: float, min_lr: float, warmup: int) -> float:
    """Linear warmup followed by cosine decay down to min_lr."""
    if it < warmup:
        return lr * (it + 1) / max(1, warmup)
    if it >= max_iters:
        return min_lr
    ratio = (it - warmup) / max(1, max_iters - warmup)
    return min_lr + 0.5 * (1 + math.cos(math.pi * ratio)) * (lr - min_lr)


def atomic_save(obj, filepath: str):
    """Atomically save an object or dictionary to disk using a temporary file."""
    os.makedirs(os.path.dirname(filepath) or ".", exist_ok=True)
    tmp_path = filepath + f".tmp_{os.getpid()}_{time.time_ns()}"
    if filepath.endswith(".json"):
        with open(tmp_path, "w", encoding="utf-8") as f:
            json.dump(obj, f, indent=2)
    else:
        torch.save(obj, tmp_path)
    os.replace(tmp_path, filepath)


@torch.no_grad()
def estimate_loss(model: GPT, ds: CharDataset, cfg: dict, device: str, amp_ctx) -> dict:
    """Computes average loss over fixed evaluation batches.
    
    Using a fixed generator seed (1234) ensures loss numbers are directly comparable
    across evaluation checkpoints.
    """
    model.eval()
    out = {}
    for split in ("train", "val"):
        g = torch.Generator().manual_seed(1234)
        losses = []
        for _ in range(cfg["eval_iters"]):
            x, y = ds.get_batch(split, cfg["batch_size"], cfg["block_size"], device, g)
            with amp_ctx:
                _, loss = model(x, y)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out


def train(**overrides) -> dict:
    """Executes a single model training run with checkpointing and metric logging."""
    cfg = {**DEFAULTS, **overrides}

    # Device selection
    if cfg["device"] is not None:
        device = cfg["device"]
    else:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    seed_everything(cfg["seed"])

    # Load corpus
    if cfg["data_text"] is not None:
        raw_text = cfg["data_text"]
    else:
        raw_text = load_text(cfg["dataset_path"])

    ds = CharDataset(raw_text)
    text_sha256 = compute_sha256(raw_text)

    # Initialize model
    mc = GPTConfig(
        block_size=cfg["block_size"],
        vocab_size=ds.vocab_size,
        n_layer=cfg["n_layer"],
        n_head=cfg["n_head"],
        n_embd=cfg["n_embd"],
        dropout=cfg["dropout"],
        ln_placement=cfg["ln_placement"],
    )
    model = GPT(mc).to(device)
    n_params = model.num_params()
    n_params_ne = model.num_params(non_embedding=True)

    # Setup mixed precision
    use_cuda = device.startswith("cuda")
    if use_cuda and torch.cuda.is_bf16_supported():
        amp_dtype = torch.bfloat16
        use_scaler = False
    elif use_cuda:
        amp_dtype = torch.float16
        use_scaler = True
    else:
        amp_dtype = torch.float32
        use_scaler = False

    amp_ctx = torch.autocast(device_type="cuda", dtype=amp_dtype) if use_cuda else nullcontext()
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

    # Optimizer: weight decay on 2D parameter tensors, no decay on 1D biases/LayerNorms
    decay = [p for p in model.parameters() if p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.dim() < 2]
    optimizer = torch.optim.AdamW(
        [
            {"params": decay, "weight_decay": cfg["weight_decay"]},
            {"params": no_decay, "weight_decay": 0.0},
        ],
        lr=cfg["lr"],
        betas=(0.9, 0.95),
    )

    start_iter = 0
    history = []
    best_val = float("inf")
    t0 = time.time()
    g_train = torch.Generator().manual_seed(cfg["seed"])

    # Handle step-level resumption if requested
    if cfg["resume_from"] is not None:
        if not os.path.isfile(cfg["resume_from"]):
            raise FileNotFoundError(f"Resume checkpoint not found: {cfg['resume_from']}")
        checkpoint = torch.load(cfg["resume_from"], map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        if use_scaler and "scaler_state_dict" in checkpoint:
            scaler.load_state_dict(checkpoint["scaler_state_dict"])
        start_iter = checkpoint["iteration"] + 1
        history = checkpoint.get("history", [])
        best_val = checkpoint.get("best_val", float("inf"))
        if "rng_state_cpu" in checkpoint:
            torch.set_rng_state(checkpoint["rng_state_cpu"].cpu())
        if use_cuda and "rng_state_cuda" in checkpoint and checkpoint["rng_state_cuda"] is not None:
            torch.cuda.set_rng_state_all([s.cpu() for s in checkpoint["rng_state_cuda"]])
        if "python_rng_state" in checkpoint:
            random.setstate(checkpoint["python_rng_state"])
        print(f"Resumed training from iteration {start_iter} (best_val: {best_val:.4f})")

    # Training loop
    for it in range(start_iter, cfg["max_iters"] + 1):
        lr = get_lr(it, cfg["max_iters"], cfg["lr"], cfg["min_lr"], cfg["warmup"])
        for group in optimizer.param_groups:
            group["lr"] = lr

        # Periodic evaluation
        if it % cfg["eval_interval"] == 0 or it == cfg["max_iters"]:
            l = estimate_loss(model, ds, cfg, device, amp_ctx)
            if l["val"] < best_val:
                best_val = l["val"]
            elapsed = time.time() - t0
            history.append(
                dict(iter=it, train=l["train"], val=l["val"], lr=lr, seconds=elapsed)
            )
            print(
                f"iter {it:5d} | train {l['train']:.4f} | val {l['val']:.4f} | lr {lr:.2e} | {elapsed:.0f}s",
                flush=True,
            )

            # Save intermediate resumable state
            if cfg["save_resume"] and it < cfg["max_iters"]:
                resume_dict = {
                    "format_version": "1.0",
                    "iteration": it,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scaler_state_dict": scaler.state_dict() if use_scaler else None,
                    "history": history,
                    "best_val": best_val,
                    "model_config": mc.to_dict(),
                    "chars": ds.chars,
                    "cfg": cfg,
                    "rng_state_cpu": torch.get_rng_state(),
                    "rng_state_cuda": torch.cuda.get_rng_state_all() if use_cuda else None,
                    "python_rng_state": random.getstate(),
                }
                atomic_save(resume_dict, os.path.join(cfg["out_dir"], "resume_state.pt"))

        if it == cfg["max_iters"]:
            break

        # Sample batch & forward pass
        x, y = ds.get_batch("train", cfg["batch_size"], cfg["block_size"], device, g_train)
        with amp_ctx:
            _, loss = model(x, y)

        if not math.isfinite(loss.item()):
            raise FloatingPointError(f"Loss became non-finite ({loss.item()}) at iteration {it}")

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
        scaler.step(optimizer)
        scaler.update()

    total_time = time.time() - t0
    final_train = history[-1]["train"] if history else float("nan")
    final_val = history[-1]["val"] if history else float("nan")
    best_val_calc = min((h["val"] for h in history), default=float("nan"))

    # Construct unique run fingerprint
    fingerprint_raw = f"{cfg['n_layer']}_{cfg['n_head']}_{cfg['n_embd']}_{cfg['ln_placement']}_{cfg['seed']}_{cfg['max_iters']}_{cfg['lr']}"
    fingerprint = hashlib.sha256(fingerprint_raw.encode("utf-8")).hexdigest()[:16]

    result = dict(
        schema_version="1.0",
        fingerprint=fingerprint,
        config=cfg,
        n_params=n_params,
        n_params_non_embedding=n_params_ne,
        dataset=dict(vocab_size=ds.vocab_size, sha256=text_sha256),
        environment=dict(
            python_version=platform.python_version(),
            torch_version=torch.__version__,
            device=device,
            cuda_available=torch.cuda.is_available(),
            device_name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        ),
        final_train=final_train,
        final_val=final_val,
        best_val=best_val_calc,
        history=history,
        seconds=total_time,
    )

    # Save final results atomically
    os.makedirs(cfg["out_dir"], exist_ok=True)
    atomic_save(result, os.path.join(cfg["out_dir"], "result.json"))

    if cfg["save_ckpt"]:
        ckpt_dict = dict(
            format_version="1.0",
            model_config=mc.to_dict(),
            model=model.state_dict(),
            chars=ds.chars,
            chars_sha256=compute_sha256("".join(ds.chars)),
            train_config=dict(
                max_iters=cfg["max_iters"],
                batch_size=cfg["batch_size"],
                lr=cfg["lr"],
                seed=cfg["seed"],
            ),
            metrics=dict(
                best_val=best_val_calc,
                final_val=final_val,
                final_train=final_train,
            ),
        )
        atomic_save(ckpt_dict, os.path.join(cfg["out_dir"], "ckpt.pt"))

    return result


if __name__ == "__main__":
    p = argparse.ArgumentParser(description="nanoGPT training CLI")
    for k, v in DEFAULTS.items():
        if isinstance(v, bool):
            p.add_argument(f"--{k}", type=lambda s: s.lower() in ("1", "true", "yes"), default=v)
        elif v is None:
            p.add_argument(f"--{k}", type=str, default=None)
        else:
            p.add_argument(f"--{k}", type=type(v), default=v)
    args = vars(p.parse_args())
    try:
        train(**args)
    except Exception as e:
        import sys
        sys.stderr.write(f"Error during training: {e}\n")
        sys.exit(1)


In [ ]:
%%writefile sweep.py
"""Scaling study: train several model sizes x seeds (plus an optional LayerNorm-placement ablation).

Preserves standard model sizes:
  xs: (2 layers, 2 heads, 64 embd)   ~0.11M parameters
  s:  (4 layers, 4 heads, 128 embd)  ~0.82M parameters
  m:  (6 layers, 6 heads, 192 embd)  ~2.71M parameters
  l:  (6 layers, 6 heads, 384 embd)  ~10.72M parameters

Validation and Resumption:
  Validates result.json integrity, schema, and hyperparameter matching before skipping a run.
  Quarantines corrupted or incomplete result files so runs are safely re-executed.
"""
import argparse
import json
import math
import os
import shutil
import time

from train import train

# Standard model configurations: (n_layer, n_head, n_embd)
SIZES = {
    "xs": (2, 2, 64),
    "s": (4, 4, 128),
    "m": (6, 6, 192),
    "l": (6, 6, 384),
}


def is_run_valid(
    result_path: str,
    expected_size: str,
    expected_ln: str,
    expected_seed: int,
    expected_max_iters: int,
) -> tuple[bool, str]:
    """Validates whether an existing result.json is complete, uncorrupted, and matches configuration."""
    if not os.path.isfile(result_path):
        return False, "file does not exist"
    if os.path.getsize(result_path) == 0:
        return False, "file is empty (0 bytes)"

    try:
        with open(result_path, encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        return False, f"JSON parse error: {e}"

    # Required top-level keys
    for req in ("config", "best_val", "final_val", "history", "n_params"):
        if req not in data:
            return False, f"Missing required key: {req!r}"

    # Verify finite loss values
    for metric in ("best_val", "final_val"):
        val = data[metric]
        if not isinstance(val, (int, float)) or not math.isfinite(val):
            return False, f"Non-finite or invalid metric: {metric}={val}"

    # Verify history completeness
    history = data.get("history", [])
    if not history or not isinstance(history, list):
        return False, "Empty or invalid history list"
    if history[-1].get("iter") != expected_max_iters:
        return False, f"Incomplete run: history ended at iter {history[-1].get('iter')}, expected {expected_max_iters}"

    # Verify configuration matching
    cfg = data.get("config", {})
    if cfg.get("ln_placement") != expected_ln:
        return False, f"ln_placement mismatch: found {cfg.get('ln_placement')}, expected {expected_ln}"
    if cfg.get("seed") != expected_seed:
        return False, f"seed mismatch: found {cfg.get('seed')}, expected {expected_seed}"

    return True, "valid"


def run_sweep(
    out: str = "results",
    sizes: list[str] | None = None,
    seeds: list[int] | None = None,
    max_iters: int = 3000,
    ablate: bool = False,
    quick: bool = False,
    force: bool = False,
    save_ckpt: bool = True,
    device: str | None = None,
    data_text: str | None = None,
    **train_overrides,
) -> list[dict]:
    """Orchestrates multi-size, multi-seed training sweep."""
    if sizes is None:
        sizes = list(SIZES.keys())
    if seeds is None:
        seeds = [0, 1, 2]

    for s in sizes:
        if s not in SIZES:
            raise ValueError(f"Unknown size: {s!r}. Supported sizes: {list(SIZES.keys())}")

    extra = {}
    if quick:
        sizes = [s for s in sizes if s in ("xs", "s")]
        seeds = [s for s in seeds if s in (0, 1)]
        max_iters = min(max_iters, 60)
        extra = dict(batch_size=16, block_size=32, eval_interval=30, eval_iters=5)

    placements = ["pre", "post"] if ablate else ["pre"]
    completed_runs = []

    for ln in placements:
        for size in sizes:
            n_layer, n_head, n_embd = SIZES[size]
            for seed in seeds:
                name = f"{size}_{ln}_seed{seed}"
                out_dir = os.path.join(out, name)
                result_path = os.path.join(out_dir, "result.json")

                if not force and os.path.exists(result_path):
                    valid, reason = is_run_valid(result_path, size, ln, seed, max_iters)
                    if valid:
                        print(f"skip {name} (already completed and verified)")
                        with open(result_path, encoding="utf-8") as f:
                            completed_runs.append(json.load(f))
                        continue
                    else:
                        corrupt_dest = result_path + f".corrupt_{int(time.time())}"
                        print(f"Quarantining corrupted result for {name} ({reason}) -> {corrupt_dest}")
                        shutil.move(result_path, corrupt_dest)

                print(f"\n=== Running {name} (size={size}, ln={ln}, seed={seed}) ===", flush=True)
                run_res = train(
                    n_layer=n_layer,
                    n_head=n_head,
                    n_embd=n_embd,
                    ln_placement=ln,
                    seed=seed,
                    max_iters=max_iters,
                    out_dir=out_dir,
                    save_ckpt=save_ckpt,
                    device=device,
                    data_text=data_text,
                    **{**extra, **train_overrides},
                )
                completed_runs.append(run_res)

    return completed_runs


if __name__ == "__main__":
    p = argparse.ArgumentParser(description="nanoGPT Scaling Study Sweep")
    p.add_argument("--out", default="results", help="Directory to store run outputs")
    p.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2], help="Random seeds to evaluate")
    p.add_argument("--sizes", nargs="+", default=list(SIZES.keys()), help="Model sizes to run")
    p.add_argument("--max_iters", type=int, default=3000, help="Training iterations per run")
    p.add_argument("--ablate", action="store_true", help="Also train Post-LN variant for each configuration")
    p.add_argument("--quick", action="store_true", help="Tiny CPU smoke test")
    p.add_argument("--force", action="store_true", help="Force rerun even if valid result exists")
    p.add_argument("--device", default=None, help="Device override ('cpu', 'cuda')")
    a = p.parse_args()

    try:
        run_sweep(
            out=a.out,
            sizes=a.sizes,
            seeds=a.seeds,
            max_iters=a.max_iters,
            ablate=a.ablate,
            quick=a.quick,
            force=a.force,
            device=a.device,
        )
    except Exception as e:
        import sys
        sys.stderr.write(f"Error during sweep: {e}\n")
        sys.exit(1)


In [ ]:
%%writefile plot.py
"""Aggregate sweep results into statistical summaries, plots, and markdown reports.

Supports:
- CLI usage: python plot.py --results results --out figures --metric best_val
- Library usage: aggregate_and_plot(...)
"""
import argparse
import glob
import json
import math
import os
import re
from collections import defaultdict

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

# Two-sided 95% Student-t critical values by degrees of freedom (df = n - 1)
T95 = {
    1: 12.706,
    2: 4.303,
    3: 3.182,
    4: 2.776,
    5: 2.571,
    6: 2.447,
    7: 2.365,
    8: 2.306,
    9: 2.262,
    10: 2.228,
    11: 2.201,
    12: 2.179,
    13: 2.160,
    14: 2.145,
    15: 2.131,
    16: 2.120,
    17: 2.110,
    18: 2.101,
    19: 2.093,
    20: 2.086,
    25: 2.060,
    30: 2.042,
}


def t_crit(df: int) -> float:
    """Student-t multiplier for 95% confidence interval (df = n - 1)."""
    if df < 1:
        return 0.0
    if df > 30:
        return 1.96
    return T95[max(k for k in T95 if k <= df)]


def compute_group_stats(vals: list[float], n_params: int, n_params_ne: int) -> dict:
    """Computes mean, sample std (ddof=1), and Student-t 95% CI half-width."""
    arr = np.array(vals, dtype=float)
    n = len(arr)
    mean = float(arr.mean())
    std = float(arr.std(ddof=1)) if n > 1 else 0.0
    crit = t_crit(n - 1)
    ci95 = float(crit * std / np.sqrt(n)) if n > 1 else 0.0
    return dict(
        n=n,
        raw_values=[round(v, 6) for v in vals],
        params=n_params,
        params_ne=n_params_ne,
        mean=round(mean, 6),
        std=round(std, 6),
        ci95=round(ci95, 6),
    )


def load_runs(results_dir: str) -> list[dict]:
    """Safely loads and validates all result.json files found in results_dir."""
    runs = []
    pattern = os.path.join(results_dir, "*", "result.json")
    for path in sorted(glob.glob(pattern)):
        try:
            with open(path, encoding="utf-8") as f:
                r = json.load(f)
        except Exception as e:
            print(f"Warning: could not parse {path}: {e}")
            continue

        dir_name = os.path.basename(os.path.dirname(path))
        # Infer size and ln placement from config or dir name
        cfg = r.get("config", {})
        ln = cfg.get("ln_placement")
        size = None

        m = re.match(r"^([a-zA-Z0-9]+)_(pre|post)_seed(\d+)$", dir_name)
        if m:
            size_candidate, ln_candidate, seed_candidate = m.group(1), m.group(2), int(m.group(3))
            size = size or size_candidate
            ln = ln or ln_candidate
            seed = cfg.get("seed", seed_candidate)
        else:
            seed = cfg.get("seed", 0)

        # Infer size if not in name
        if not size:
            n_layer = cfg.get("n_layer")
            n_embd = cfg.get("n_embd")
            if (n_layer, n_embd) == (2, 64):
                size = "xs"
            elif (n_layer, n_embd) == (4, 128):
                size = "s"
            elif (n_layer, n_embd) == (6, 192):
                size = "m"
            elif (n_layer, n_embd) == (6, 384):
                size = "l"
            else:
                size = f"L{n_layer}E{n_embd}"

        ln = ln or "pre"
        r.update(size=size, ln=ln, name=dir_name, seed=seed)
        runs.append(r)
    return runs


def aggregate_and_plot(results_dir: str = "results", out_dir: str = "figures", metric: str = "best_val") -> dict:
    """Aggregates metrics, computes Student-t statistics, and generates figures and summaries."""
    os.makedirs(out_dir, exist_ok=True)
    runs = load_runs(results_dir)
    if not runs:
        raise ValueError(f"No valid result.json files found in '{results_dir}'")

    # Group by (ln_placement, size)
    groups = defaultdict(list)
    for r in runs:
        if metric in r and math.isfinite(r[metric]):
            groups[(r["ln"], r["size"])].append(r)

    stats = {}
    for key, rs in groups.items():
        vals = [r[metric] for r in rs]
        n_params = rs[0].get("n_params", 0)
        n_params_ne = rs[0].get("n_params_non_embedding", 0)
        stats[key] = compute_group_stats(vals, n_params, n_params_ne)

    # 1. Scaling Plot
    fig, ax = plt.subplots(figsize=(7, 5))
    slope_text = ""
    for ln, marker in (("pre", "o"), ("post", "s")):
        ks = sorted([k for k in stats if k[0] == ln], key=lambda k: stats[k]["params"])
        if not ks:
            continue
        x = np.array([stats[k]["params"] for k in ks], dtype=float)
        y = np.array([stats[k]["mean"] for k in ks])
        e = np.array([stats[k]["std"] for k in ks])

        ax.errorbar(
            x,
            y,
            yerr=e,
            marker=marker,
            capsize=4,
            linewidth=1.5,
            label=f"{ln}-LN (mean \u00b1 sample std, ddof=1)",
        )

        if ln == "pre" and len(x) >= 2 and all(y > 0):
            slope, intercept = np.polyfit(np.log10(x), np.log10(y), 1)
            xs = np.geomspace(x.min(), x.max(), 50)
            ys = 10 ** (intercept + slope * np.log10(xs))
            ax.plot(xs, ys, "--", alpha=0.7, label=f"Pre-LN log-log fit (slope {slope:.3f})")
            slope_text = f"Pre-LN log-log validation loss scaling slope: {slope:.4f}\n"

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Total Parameters (log scale)")
    ax.set_ylabel(f"Validation Loss [{metric}] (nats/char, log scale)")
    ax.set_title("Model Size Scaling: Validation Loss vs. Parameter Count")
    ax.legend()
    ax.grid(True, which="both", alpha=0.3, linestyle=":")
    fig.tight_layout()
    scaling_plot_path = os.path.join(out_dir, "scaling.png")
    fig.savefig(scaling_plot_path, dpi=150)
    plt.close(fig)

    # 2. Validation Loss Curves Plot
    fig, ax = plt.subplots(figsize=(7, 5))
    curves_plotted = 0
    for r in sorted(runs, key=lambda x: (x.get("n_params", 0), x.get("seed", 0))):
        if r["ln"] == "pre" and r.get("seed", 0) == 0 and "history" in r:
            iters = [h["iter"] for h in r["history"] if "iter" in h and "val" in h]
            val_losses = [h["val"] for h in r["history"] if "iter" in h and "val" in h]
            if iters and val_losses:
                label = f"{r['size']} ({r.get('n_params', 0)/1e6:.2f}M params)"
                ax.plot(iters, val_losses, label=label, linewidth=1.5)
                curves_plotted += 1

    ax.set_xlabel("Training Iteration")
    ax.set_ylabel("Validation Loss (nats/char)")
    ax.set_title("Validation Loss Trajectory During Training (Seed 0, Pre-LN)")
    if curves_plotted > 0:
        ax.legend()
    ax.grid(True, alpha=0.3, linestyle=":")
    fig.tight_layout()
    curves_plot_path = os.path.join(out_dir, "curves.png")
    fig.savefig(curves_plot_path, dpi=150)
    plt.close(fig)

    # 3. Markdown Summary Table
    md_lines = [
        f"# Scaling Study Summary ({metric})",
        "",
        "> **Note on Metrics & Uncertainty:**",
        f"> - Metric reported: `{metric}`.",
        "> - `best_val` is the minimum validation loss across periodic evaluation checkpoints (an early-stopping proxy, not an unbiased test-set metric).",
        "> - Standard deviation is computed with sample degrees of freedom (ddof=1).",
        "> - Student-t 95% Confidence Interval half-widths use two-sided critical values (t=4.303 for n=3, df=2).",
        "> - Note: 3 seeds provide preliminary sample dispersion; error intervals are rough approximations.",
        "",
        "| LN Placement | Size | Parameters | Non-Embedding Params | N Seeds | Raw Values | Mean Val Loss | Std Dev (ddof=1) | 95% CI Half-Width |",
        "|:---:|:---:|---:|---:|:---:|:---|---:|---:|---:|",
    ]

    for k in sorted(stats, key=lambda k: (k[0], stats[k]["params"])):
        s = stats[k]
        raw_str = ", ".join(f"{v:.4f}" for v in s["raw_values"])
        md_lines.append(
            f"| {k[0]} | {k[1]} | {s['params']:,} | {s['params_ne']:,} | {s['n']} | [{raw_str}] | {s['mean']:.4f} | {s['std']:.4f} | \u00b1{s['ci95']:.4f} |"
        )

    md_lines.append("")
    if slope_text:
        md_lines.append(f"**Empirical Scaling Slope:** {slope_text.strip()}")
        md_lines.append(
            "> *Caution: The log-log linear fit is descriptive empirical evidence across 4 small model configurations on Tiny Shakespeare; it does not constitute proof of an asymptotic or universal scaling law.*"
        )
    md_lines.append("")

    summary_md_path = os.path.join(out_dir, "summary.md")
    with open(summary_md_path, "w", encoding="utf-8") as f:
        f.write("\n".join(md_lines))

    # 4. Machine-Readable Summary JSON
    export_stats = {f"{k[0]}_{k[1]}": v for k, v in stats.items()}
    summary_json_path = os.path.join(out_dir, "summary.json")
    with open(summary_json_path, "w", encoding="utf-8") as f:
        json.dump({"metric": metric, "stats": export_stats}, f, indent=2)

    return {"stats": export_stats, "summary_md": summary_md_path, "summary_json": summary_json_path}


if __name__ == "__main__":
    p = argparse.ArgumentParser(description="nanoGPT Results Aggregation and Plotting")
    p.add_argument("--results", default="results", help="Directory containing run results")
    p.add_argument("--out", default="figures", help="Directory to save figures and summaries")
    p.add_argument(
        "--metric",
        choices=["best_val", "final_val"],
        default="best_val",
        help="Metric to aggregate: 'best_val' (early stopping proxy) or 'final_val'",
    )
    args = p.parse_args()

    try:
        res = aggregate_and_plot(results_dir=args.results, out_dir=args.out, metric=args.metric)
        with open(res["summary_md"], encoding="utf-8") as f:
            print(f.read())
    except Exception as e:
        import sys
        sys.stderr.write(f"Error: {e}\n")
        sys.exit(1)


In [ ]:
%%writefile sample.py
"""Generate text from a saved checkpoint via CLI.

Usage:
  python sample.py --ckpt out/run/ckpt.pt --prompt "ROMEO:" --tokens 200 --temperature 0.8 --top_k 40
"""
import argparse
import os

from service import ModelService


def main():
    p = argparse.ArgumentParser(description="nanoGPT text generation CLI")
    p.add_argument("--ckpt", default="out/run/ckpt.pt", help="Path to checkpoint (.pt) file")
    p.add_argument("--prompt", default="\n", help="Initial prompt text")
    p.add_argument("--tokens", type=int, default=200, help="Number of new tokens/characters to generate")
    p.add_argument("--temperature", type=float, default=0.8, help="Softmax sampling temperature")
    p.add_argument("--top_k", type=int, default=40, help="Top-k filtering limit")
    p.add_argument("--seed", type=int, default=0, help="Random seed for generation")
    p.add_argument("--device", default=None, help="Device ('cpu', 'cuda')")
    a = p.parse_args()

    if not os.path.isfile(a.ckpt):
        import sys
        sys.stderr.write(f"Error: Checkpoint not found at '{a.ckpt}'. Please train a model first.\n")
        sys.exit(1)

    try:
        service = ModelService(default_device=a.device)
        service.registry.register_manual("cli_model", a.ckpt)

        res = service.generate_text(
            model_id="cli_model",
            prompt=a.prompt,
            max_new_tokens=a.tokens,
            temperature=a.temperature,
            top_k=a.top_k,
            seed=a.seed,
        )
        print(res["full_text"])
    except Exception as e:
        import sys
        sys.stderr.write(f"Error: {e}\n")
        sys.exit(1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile tests/test_model.py
"""Sanity, architecture, and correctness tests for model.py.
Run:  pytest -q
"""
import math
import os
import sys

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
import pytest
import torch

from model import GPT, GPTConfig


def make(ln="pre", **kw):
    cfg = GPTConfig(
        block_size=16,
        vocab_size=65,
        n_layer=2,
        n_head=2,
        n_embd=32,
        dropout=0.0,
        ln_placement=ln,
        **kw,
    )
    return GPT(cfg)


def test_shapes():
    """Verify forward output shapes for both Pre-LN and Post-LN architectures."""
    for ln in ("pre", "post"):
        m = make(ln)
        x = torch.randint(0, 65, (3, 16))
        logits, loss = m(x, x)
        assert logits.shape == (3, 16, 65)
        assert loss.ndim == 0


def test_causality():
    """Causal attention invariant: changing the LAST token must not alter logits at earlier positions."""
    torch.manual_seed(0)
    m = make().eval()
    x = torch.randint(0, 65, (2, 16))
    x2 = x.clone()
    x2[:, -1] = (x2[:, -1] + 1) % 65
    l1, _ = m(x)
    l2, _ = m(x2)
    assert torch.allclose(l1[:, :-1], l2[:, :-1], atol=1e-5)
    assert not torch.allclose(l1[:, -1], l2[:, -1], atol=1e-5)


def test_initial_loss_is_near_uniform():
    """Verify unlearned model produces cross-entropy loss close to uniform random guessing ln(V).
    
    CRITICAL: Targets 'y' must be sampled independently of inputs 'x'. If targets are identical to
    inputs (m(x, x)), weight tying (wte == head) induces a positive dot-product bias on the target
    logit, which artificially depresses initial cross entropy below uniform ln(V).
    """
    torch.manual_seed(0)
    m = make().eval()
    x = torch.randint(0, 65, (8, 16))
    # Independent random targets:
    y = torch.randint(0, 65, (8, 16))
    _, loss = m(x, y)
    expected_uniform_loss = math.log(65)  # ~4.1744
    assert abs(loss.item() - expected_uniform_loss) < 0.3, (
        f"Initial loss {loss.item():.4f} diverged from uniform ln(65)={expected_uniform_loss:.4f}"
    )


def test_weight_tying_downward_bias_regression():
    """Regression test documenting why m(x, x) is an invalid uniform-loss test fixture.
    
    Because token embeddings (wte) and output projection (head) are tied, passing identical input
    and target tokens m(x, x) creates a positive correlation between x and the target logit,
    depressing loss below ln(65).
    """
    torch.manual_seed(0)
    m = make().eval()
    x = torch.randint(0, 65, (8, 16))
    _, loss_same = m(x, x)
    expected_uniform_loss = math.log(65)
    # The loss for identical input/target is biased downwards (> 0.3 below ln(65))
    assert expected_uniform_loss - loss_same.item() > 0.3, (
        f"Expected downward bias from tied weights, got loss={loss_same.item()}"
    )


def test_can_overfit_one_batch():
    """Ensure gradients flow properly and a small batch can be overfit."""
    torch.manual_seed(0)
    m = make()
    x = torch.randint(0, 65, (4, 16))
    y = torch.roll(x, -1, dims=1)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-3)
    _, first = m(x, y)
    for _ in range(150):
        _, loss = m(x, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    assert loss.item() < 0.5 * first.item(), (first.item(), loss.item())


def test_generate_length():
    """Verify autoregressive generation yields the requested sequence length."""
    m = make().eval()
    out = m.generate(torch.zeros((1, 1), dtype=torch.long), 20, top_k=10)
    assert out.shape == (1, 21)


def test_parameter_counts_and_ln_difference():
    """Verify exact parameter counts for XS, S, M, L against documented formulas.
    
    Verifies that weight tying is respected and Post-LN has exactly 2*n_embd fewer parameters
    due to omitting the final LayerNorm.
    """
    sizes = {
        "xs": (2, 2, 64, 112448),
        "s": (4, 4, 128, 818048),
        "m": (6, 6, 192, 2706624),
        "l": (6, 6, 384, 10721664),
    }
    for name, (n_layer, n_head, n_embd, expected_params) in sizes.items():
        cfg_pre = GPTConfig(
            block_size=128, vocab_size=65, n_layer=n_layer, n_head=n_head, n_embd=n_embd, ln_placement="pre"
        )
        m_pre = GPT(cfg_pre)
        assert m_pre.num_params() == expected_params, (
            f"{name} Pre-LN params {m_pre.num_params()} != {expected_params}"
        )

        cfg_post = GPTConfig(
            block_size=128, vocab_size=65, n_layer=n_layer, n_head=n_head, n_embd=n_embd, ln_placement="post"
        )
        m_post = GPT(cfg_post)
        # Post-LN omits final LayerNorm (weight + bias = 2*n_embd params)
        expected_post_params = expected_params - 2 * n_embd
        assert m_post.num_params() == expected_post_params, (
            f"{name} Post-LN params {m_post.num_params()} != {expected_post_params}"
        )


def test_config_validation():
    """Verify invalid configurations raise descriptive ValueError."""
    with pytest.raises(ValueError, match="block_size must be positive"):
        GPTConfig(block_size=0)
    with pytest.raises(ValueError, match="vocab_size must be positive"):
        GPTConfig(vocab_size=-1)
    with pytest.raises(ValueError, match="divisible by n_head"):
        GPTConfig(n_embd=65, n_head=4)
    with pytest.raises(ValueError, match="dropout must be in"):
        GPTConfig(dropout=1.5)
    with pytest.raises(ValueError, match="ln_placement must be 'pre' or 'post'"):
        GPTConfig(ln_placement="middle")


def test_forward_and_generate_error_handling():
    """Verify sequence length checks and sampling boundary validations."""
    m = make().eval()
    # Sequence length exceeding block_size
    too_long = torch.randint(0, 65, (1, 17))
    with pytest.raises(ValueError, match="exceeds model block_size"):
        m(too_long)

    # Targets shape mismatch
    x = torch.randint(0, 65, (2, 16))
    bad_targets = torch.randint(0, 65, (2, 10))
    with pytest.raises(ValueError, match="targets shape"):
        m(x, bad_targets)

    # Invalid generate parameters
    with pytest.raises(ValueError, match="temperature must be positive"):
        m.generate(x, 10, temperature=0.0)
    with pytest.raises(ValueError, match="max_new_tokens must be non-negative"):
        m.generate(x, -5)
    with pytest.raises(ValueError, match="top_k must be positive"):
        m.generate(x, 5, top_k=0)


if __name__ == "__main__":
    os.environ["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"
    sys.exit(pytest.main([__file__, "-v"]))


## 2. Sanity tests (run these first)
All five must print PASS. **If any fails, do not train: debug first.**

In [ ]:
!python tests/test_model.py
!python -m pytest -q


## 3. Quick training run (a few minutes on a T4)
Watch the validation loss fall. It should start near 4.17 (= ln 65) and drop well below 2.

In [ ]:
from train import train
r = train(max_iters=1000, eval_interval=100, out_dir='out/quick')
print('final val loss:', round(r['final_val'], 4), '| params:', r['n_params'])

In [ ]:
!python sample.py --ckpt out/quick/ckpt.pt --prompt "ROMEO:" --tokens 300

**Question for you:** is the generated text good? What kinds of mistakes does it make, and why would a larger model or longer training help?

## 4. Scaling sweep
4 sizes x 3 seeds (about 12 runs). Add `--ablate` to also run post-LN variants (doubles the time). Time depends on the GPU Colab gives you.

In [ ]:
!python sweep.py --out $RESULTS_DIR

In [ ]:
# Optional ablation (post-LN). Uncomment to run:
# !python sweep.py --out $RESULTS_DIR --ablate

## 5. Plots and summary table
`--metric best_val` (default) uses the lowest validation loss over the evaluation checkpoints, which guards against overfitting in the larger models. Re-run with `--metric final_val` to compare.

In [ ]:
!python plot.py --results $RESULTS_DIR --out figures

In [ ]:
from IPython.display import Image, display
display(Image('figures/scaling.png'))
display(Image('figures/curves.png'))

In [ ]:
print(open('figures/summary.md').read())

## 6. Download your results
Zip the figures and results so you can add them to your GitHub repo.

In [ ]:
!zip -r my_results.zip figures $RESULTS_DIR -x '*.pt'
from google.colab import files
files.download('my_results.zip')

## What to write up
Use the template in README.md: question, setup, results table with confidence intervals, what surprised you, what you got wrong, how to reproduce.